# ReviewSense PT-BR — 01 · Auditoria e EDA

**Fase 1:** carregar, validar, auditar, limpar e dividir B2W-Reviews01.
Não treina modelos. As regras de processamento vivem em `src/reviewsense/`.

**Fonte:** B2W Digital, avaliações da Americanas.com (jan–mai/2018),
[repositório original](https://github.com/americanas-tech/b2w-reviews01),
**CC BY-NC-SA 4.0**. As transformações estão descritas no README e no relatório
agregado. A licença MIT do código não altera a licença dos dados.

Os rótulos são proxies derivadas de estrelas, não anotações humanas de sentimento.
Não exiba avaliações brutas ou metadados pessoais em saídas que serão publicadas.

## Preparação local (principal) e Google Colab (alternativa)

Local: instale `python -m pip install -r requirements-dev.txt` na raiz e selecione
o kernel desse ambiente. Execute as células em ordem. A Fase 1 usa CPU.
A GPU local será considerada nas fases de modelos, após verificar seu suporte.
Colab é opcional, caso memória ou tempo de execução locais se tornem limitantes.

Colab: use uma sessão CPU. Antes de executar as células abaixo, disponibilize o
repositório completo em `/content/Review-Sense` (clone após publicar os commits no
remoto, ou transfira a pasta). Em uma célula de preparação, execute:

```python
!git clone https://github.com/jorgeribeiru/Review-Sense.git /content/Review-Sense
%cd /content/Review-Sense
%pip install -r requirements-dev.txt
```

Se a pasta já existir, reutilize-a em vez de cloná-la novamente. Para outro caminho,
defina `os.environ["REVIEWSENSE_REPO"]` antes da próxima célula.
Os dados são baixados apenas para a sessão local. `/content` é efêmero: copie os
artefatos que quiser preservar para armazenamento persistente.

In [ ]:
import json
import os
from pathlib import Path

candidates = [Path(os.environ["REVIEWSENSE_REPO"])] if "REVIEWSENSE_REPO" in os.environ else [
    Path.cwd(), Path.cwd().parent, Path("/content/Review-Sense")
]
ROOT = next((path.resolve() for path in candidates
             if (path / "src/reviewsense/data.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Disponibilize o repositório e defina REVIEWSENSE_REPO; veja acima.")
os.chdir(ROOT)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from reviewsense.data import (
    audit_dataset, build_manifest, load_b2w, prepare_dataset,
    save_artifacts, split_dataset, validate_splits,
)
from reviewsense.labels import LABEL_NAMES

config = json.loads((ROOT / "configs/data.json").read_text(encoding="utf-8"))
config

## 1. Carregamento e contrato

Baixar a revisão oficial fixada e validar SHA-256, colunas, notas e tipos dos textos.
O carregador reutiliza o cache verificado; não ignora silenciosamente erros.
O CSV real contém metadados que serão excluídos dos dados preparados.

In [ ]:
raw = load_b2w(cache_dir=ROOT / "data/raw")
audit = audit_dataset(raw)
display(pd.Series({"linhas": len(raw), "colunas": len(raw.columns)}))
display(pd.DataFrame({"dtype": audit["dtypes"], "ausentes": audit["missing_values"]}))
raw.attrs["source"]

## 2. Distribuições e qualidade dos textos

Contagens abaixo incluem os dados brutos; os comprimentos são medidos após
normalização e excluem vazios. Palavras são contadas por separação em espaços,
sem aplicar tokenizer. Nenhuma avaliação individual é exibida.

In [ ]:
display(pd.DataFrame(audit["class_distribution"]).T)
display(pd.DataFrame({
    "caracteres": audit["text_characters"],
    "palavras_por_espacos": audit["text_words_whitespace"],
}))
quality_keys = [
    "empty_texts_including_missing", "duplicate_rows_excess", "duplicate_text_groups",
    "duplicate_text_rows_excess", "conflicting_label_groups", "conflicting_label_rows",
    "conflicting_rating_groups",
]
display(pd.Series({key: audit[key] for key in quality_keys}, name="contagem"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series(audit["rating_distribution"]).plot.bar(ax=axes[0], rot=0, color="#257981")
axes[0].set(title="Notas originais", xlabel="Estrelas", ylabel="Avaliações")
pd.Series(audit["missing_values"]).sort_values().plot.barh(ax=axes[1], color="#ad6540")
axes[1].set(title="Ausentes por coluna", xlabel="Valores ausentes")
fig.tight_layout()
plt.show()

## 3. Limpeza conservadora e exclusões

Unicode NFC, controles ASCII e espaços; preservação de acentos, negações, caixa,
emojis e pontuação. Remover vazios, todos os grupos com classes conflitantes e
repetições de textos normalizados. Notas diferentes na mesma classe não são um
conflito de classe. Nenhuma discrepância semântica texto-nota é corrigida.
A saída contém apenas `sample_id`, `review_text` e `label`.

In [ ]:
prepared, cleaning = prepare_dataset(raw)
audit["cleaning"] = cleaning
display(pd.Series(cleaning, name="limpeza"))
assert set(prepared.columns) == {"sample_id", "review_text", "label"}
assert not prepared["sample_id"].duplicated().any()

## 4. Splits reproduzíveis e validação de integridade

Estratificação por classe, aproximadamente 70/15/15 e semente 42. Ordenação por
hash antes da divisão torna a saída independente da ordem do CSV.
Textos normalizados idênticos não atravessam conjuntos; quase duplicatas,
autores e produtos compartilhados não estão cobertos por essa garantia.
Após congelar o protocolo, reserve o teste para a avaliação final dos modelos.

In [ ]:
splits = split_dataset(prepared, **config)
validate_splits(splits, expected=prepared)
manifest = build_manifest(prepared, splits, source=raw.attrs["source"], config=config)

split_summary = pd.DataFrame({
    name: {"amostras": len(part), "fracao": len(part) / len(prepared),
           **{LABEL_NAMES[label]: int((part["label"] == label).sum())
              for label in LABEL_NAMES}}
    for name, part in splits.items()
}).T
display(split_summary)
display(pd.Series({name: part["fingerprint"] for name, part in manifest["splits"].items()},
                  name="fingerprints"))

In [ ]:
distribution = pd.DataFrame({
    name: {label: info["fraction"] for label, info in part["class_distribution"].items()}
    for name, part in manifest["splits"].items()
})
ax = distribution.plot.bar(figsize=(8, 4), rot=0)
ax.set(title="Distribuição das classes por conjunto", xlabel="Classe", ylabel="Proporção")
plt.tight_layout()
plt.show()

## 5. Exportação local opcional

Ative `EXPORT = True` para salvar os CSVs, auditoria e manifest com IDs/fingerprints.
O destino precisa estar ausente: uma execução congelada não é sobrescrita.
Dados e manifests com IDs ficam ignorados pelo Git; não publique textos sem revisão.
As células anteriores já validam os splits mesmo sem exportação.

In [ ]:
EXPORT = False
OUTPUT_DIR = ROOT / "data/processed/notebook-v1"
if EXPORT:
    save_artifacts(OUTPUT_DIR, splits=splits, manifest=manifest, audit=audit)
    print(f"Artefatos salvos em {OUTPUT_DIR}")
else:
    print("Exportação desativada; dados e splits validados em memória.")

## Limitações e próxima fase

- Corpus de produtos de 2018; outras épocas e setores precisam de avaliação própria.
- Estrelas são supervisão indireta; três estrelas podem representar opiniões mistas.
- A exclusão de conflitos e duplicatas modifica a distribuição e está contabilizada.
- A normalização não anonimiza nomes, contatos ou outros dados presentes nos textos.
- Auditoria semântica de discrepâncias e conteúdo sensível exige revisão humana local.
- O notebook não ajusta TF-IDF, tokenizer ou modelo. Treinamento e demo ficam para
  fases futuras; não há métricas de modelo nesta entrega.
- A validação local atende à Fase 1. Colab é opcional e só precisará de validação
  se for adotado; a GPU local ainda não foi validada nesta fase de dados.